# Trading 212 Portfolio Analysis

Comprehensive analysis of your Trading 212 broker portfolio with performance metrics, risk assessment, and actionable trading recommendations.

**Last Updated**: 2026-06-04

## Overview

This notebook provides:
- Portfolio performance metrics and analytics
- Asset allocation and diversification analysis
- Risk assessment
- Trading signals and recommendations
- Visual portfolio composition analysis
- Action plan for portfolio optimization

## 1. Import Required Libraries and Authentication

In [ ]:
# Import required libraries
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Import our custom modules
from src.api_client import Trading212Client
from src.portfolio import PortfolioAnalyzer
from src.risk_analysis import RiskAnalyzer
from src.trading_signals import SignalGenerator
from src.alerts import PriceAlertManager

# Set plot style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")
print(f"Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Initialize Trading 212 API Client
try:
    client = Trading212Client()
    print("✓ Successfully authenticated with Trading 212 API")
except ValueError as e:
    print(f"❌ Authentication Error: {e}")
    print("\nPlease create a .env file with your API key:")
    print("TRADING212_API_KEY=your_api_key_here")

## 2. Fetch Portfolio Data from Trading 212 API

Retrieve current portfolio holdings, cash balance, and position details from the Trading 212 API.

In [ ]:
# Get accounts
try:
    accounts = client.get_accounts()
    print(f"✓ Found {len(accounts)} account(s)")
    
    for acc in accounts:
        print(f"  - Account: {acc.get('id', 'N/A')}")
    
    # Use first account
    if accounts:
        account_id = accounts[0]["id"]
        account_name = accounts[0].get("name", "Main Account")
        print(f"\n✓ Using account: {account_name} ({account_id})")
    else:
        raise ValueError("No accounts found")
except Exception as e:
    print(f"❌ Error fetching accounts: {e}")

In [ ]:
# Fetch portfolio data
try:
    portfolio_data = client.get_portfolio(account_id)
    cash_data = client.get_cash(account_id)
    
    print("✓ Portfolio data fetched successfully")
    print(f"✓ Cash balance fetched")
    print(f"\nCash Summary:")
    print(f"  Free: ${cash_data.get('free', 0):,.2f}")
    print(f"  Blocked: ${cash_data.get('blocked', 0):,.2f}")
    
    # Show portfolio positions
    if 'positions' in portfolio_data:
        print(f"\n✓ Active Positions: {len(portfolio_data['positions'])}")
except Exception as e:
    print(f"❌ Error fetching portfolio data: {e}")
    portfolio_data = {"positions": []}
    cash_data = {"free": 0, "blocked": 0}

## 3. Calculate Portfolio Performance Metrics

Compute key metrics including total return, gain/loss, and performance indicators.

In [ ]:
# Initialize portfolio analyzer
analyzer = PortfolioAnalyzer(portfolio_data, cash_data)

# Get portfolio summary
portfolio_summary = analyzer.get_portfolio_summary()

print("=" * 60)
print("PORTFOLIO SUMMARY")
print("=" * 60)

for key, value in portfolio_summary.items():
    if isinstance(value, float):
        if key.endswith('_percent'):
            print(f"{key.replace('_', ' ').title()}: {value:,.2f}%")
        else:
            print(f"{key.replace('_', ' ').title()}: ${value:,.2f}")
    else:
        print(f"{key.replace('_', ' ').title()}: {value}")

print("=" * 60)

In [ ]:
# Get performance metrics
performance_metrics = analyzer.get_performance_metrics()

print("\n" + "=" * 60)
print("PERFORMANCE METRICS")
print("=" * 60)

if performance_metrics:
    for key, value in performance_metrics.items():
        if isinstance(value, float):
            print(f"{key.replace('_', ' ').title()}: {value:,.2f}%")
        else:
            print(f"{key.replace('_', ' ').title()}: {value}")
else:
    print("No positions to analyze yet")

print("=" * 60)

# Display top holdings
if not analyzer.df.empty:
    print("\n" + "=" * 60)
    print("TOP HOLDINGS")
    print("=" * 60)
    top_holdings = analyzer.get_top_holdings(5)
    print(top_holdings.to_string())
    print("=" * 60)

## 4. Analyze Asset Allocation and Diversification

In [ ]:
# Risk Analysis
risk_analyzer = RiskAnalyzer(analyzer.df, cash_data.get('free', 0))

print("\n" + "=" * 60)
print("RISK ANALYSIS")
print("=" * 60)

# Concentration Risk
concentration = risk_analyzer.get_concentration_risk()
print("\nCONCENTRATION RISK:")
for key, value in concentration.items():
    if isinstance(value, float):
        print(f"  {key.replace('_', ' ').title()}: {value:,.2f}")
    else:
        print(f"  {key.replace('_', ' ').title()}: {value}")

# Diversification
diversification = risk_analyzer.get_diversification_metrics()
print("\nDIVERSIFICATION METRICS:")
for key, value in diversification.items():
    if isinstance(value, float):
        print(f"  {key.replace('_', ' ').title()}: {value:,.2f}")
    else:
        print(f"  {key.replace('_', ' ').title()}: {value}")

# Liquidity
liquidity = risk_analyzer.get_liquidity_analysis()
print("\nLIQUIDITY ANALYSIS:")
for key, value in liquidity.items():
    if isinstance(value, float):
        print(f"  {key.replace('_', ' ').title()}: {value:,.2f}")
    else:
        print(f"  {key.replace('_', ' ').title()}: {value}")

print("\n" + "=" * 60)

## 5. Generate Buy/Sell Signals and Trading Recommendations

In [ ]:
# Generate Trading Signals
signal_gen = SignalGenerator(analyzer.df, portfolio_summary.get('total_value', 0))
all_signals = signal_gen.get_all_signals()

print("\n" + "=" * 60)
print("TRADING SIGNALS & RECOMMENDATIONS")
print("=" * 60)

# Rebalancing signals
if all_signals['rebalance']:
    print("\n🔄 REBALANCING SIGNALS:")
    for signal in all_signals['rebalance']:
        print(f"  {signal['asset_class'].upper()}: {signal['action']}")
        print(f"    - Target: {signal['target_weight']:.1%}")
        print(f"    - Current: {signal['current_weight']:.1%}")
        print(f"    - Deviation: {signal['deviation']:.1%}")
else:
    print("\n🔄 REBALANCING SIGNALS: Portfolio is well-balanced")

# Momentum signals
if all_signals['momentum']:
    print("\n📈 MOMENTUM SIGNALS:")
    for signal in all_signals['momentum']:
        print(f"  {signal['ticker']}: {signal['action']}")
        print(f"    - Momentum: {signal['momentum_strength']:.2f}%")
else:
    print("\n📈 MOMENTUM SIGNALS: No strong momentum detected")

# Risk reduction signals
if all_signals['risk_reduction']:
    print("\n⚠️ RISK REDUCTION SIGNALS:")
    for signal in all_signals['risk_reduction']:
        print(f"  {signal['ticker']}: {signal['action']}")
        print(f"    - Reason: {signal['reason']}")
else:
    print("\n⚠️ RISK REDUCTION SIGNALS: Portfolio risk is acceptable")

print("\n" + "=" * 60)

## 6. Visualize Portfolio Composition and Performance

In [ ]:
# Visualizations - Portfolio Composition
if not analyzer.df.empty:
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Portfolio allocation by value
    ax1 = axes[0, 0]
    top_holdings = analyzer.get_top_holdings(10)
    if not top_holdings.empty:
        colors = plt.cm.Set3(range(len(top_holdings)))
        wedges, texts, autotexts = ax1.pie(
            top_holdings['value'],
            labels=top_holdings['ticker'],
            autopct='%1.1f%%',
            colors=colors,
            startangle=90
        )
        ax1.set_title('Portfolio Allocation (Top 10 Holdings)', fontsize=12, fontweight='bold')
        for autotext in autotexts:
            autotext.set_color('white')
            autotext.set_fontweight('bold')
    
    # 2. Risk-return scatter
    ax2 = axes[0, 1]
    if 'ppl_percent' in analyzer.df.columns and 'value' in analyzer.df.columns:
        sizes = (analyzer.df['value'] / analyzer.df['value'].max() * 300) + 50
        colors = ['green' if x > 0 else 'red' for x in analyzer.df['ppl_percent']]
        ax2.scatter(range(len(analyzer.df)), analyzer.df['ppl_percent'], 
                   s=sizes, c=colors, alpha=0.6, edgecolors='black')
        ax2.axhline(y=0, color='black', linestyle='--', alpha=0.3)
        ax2.set_xlabel('Position Index')
        ax2.set_ylabel('Return %')
        ax2.set_title('Position Performance (Bubble = Size)', fontsize=12, fontweight='bold')
        ax2.grid(True, alpha=0.3)
    
    # 3. Cash vs. Invested
    ax3 = axes[1, 0]
    invested = portfolio_summary['total_value']
    cash = portfolio_summary.get('cash_balance', 0)
    labels = ['Invested', 'Cash']
    sizes = [invested, cash]
    colors = ['#ff9999', '#66b3ff']
    ax3.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
    ax3.set_title('Asset Allocation (Invested vs. Cash)', fontsize=12, fontweight='bold')
    
    # 4. Top performers
    ax4 = axes[1, 1]
    if 'ppl_percent' in analyzer.df.columns:
        top_perf = analyzer.df.nlargest(5, 'ppl_percent')[['ticker', 'ppl_percent']]
        colors_perf = ['green' if x > 0 else 'red' for x in top_perf['ppl_percent']]
        ax4.barh(range(len(top_perf)), top_perf['ppl_percent'], color=colors_perf)
        ax4.set_yticks(range(len(top_perf)))
        ax4.set_yticklabels(top_perf['ticker'])
        ax4.set_xlabel('Return %')
        ax4.set_title('Top Performers', fontsize=12, fontweight='bold')
        ax4.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
        ax4.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    print("✓ Portfolio visualizations generated")
else:
    print("No positions to visualize")

## 7. Create Action Plan and Next Steps

In [ ]:
# Generate Action Plan
print("\n" + "=" * 60)
print("RECOMMENDED ACTION PLAN")
print("=" * 60)

action_plan = []
priority_counter = 1

# 1. Risk Management Actions
if concentration.get('risk_level') == "High (Concentrated)":
    action_plan.append({
        'priority': priority_counter,
        'action': 'Reduce Concentration Risk',
        'details': f"Portfolio is concentrated with top position at {concentration['top_position_weight']:.1f}%",
        'recommended': 'Diversify by selling 10-20% of largest positions',
        'timeline': 'Within 1-2 weeks'
    })
    priority_counter += 1

# 2. Liquidity Actions
if liquidity['liquidity_level'] == "Low":
    action_plan.append({
        'priority': priority_counter,
        'action': 'Increase Cash Position',
        'details': f"Cash ratio is only {liquidity['cash_ratio_percent']:.1f}%",
        'recommended': 'Build up to 10-20% cash for opportunities',
        'timeline': 'Gradual, over 2-4 weeks'
    })
    priority_counter += 1

# 3. Performance Actions
if performance_metrics and performance_metrics.get('positions_in_loss', 0) > 0:
    action_plan.append({
        'priority': priority_counter,
        'action': 'Review Loss-Making Positions',
        'details': f"{performance_metrics['positions_in_loss']} positions are in loss",
        'recommended': 'Assess fundamentals and consider stop-losses',
        'timeline': 'Immediate review'
    })
    priority_counter += 1

# 4. Momentum Actions
if all_signals['momentum']:
    action_plan.append({
        'priority': priority_counter,
        'action': 'Support Momentum Positions',
        'details': f"Found {len(all_signals['momentum'])} momentum positions",
        'recommended': 'Consider holding or adding to strong performers',
        'timeline': 'Ongoing monitoring'
    })
    priority_counter += 1

# 5. Rebalancing Actions
if all_signals['rebalance']:
    action_plan.append({
        'priority': priority_counter,
        'action': 'Rebalance Portfolio',
        'details': f"Found {len(all_signals['rebalance'])} rebalancing opportunities",
        'recommended': 'Execute rebalancing trades',
        'timeline': 'Within 1 week'
    })
    priority_counter += 1

# Default action if no issues
if not action_plan:
    action_plan.append({
        'priority': 1,
        'action': 'Maintain Current Strategy',
        'details': 'Portfolio metrics are healthy',
        'recommended': 'Continue monitoring and hold positions',
        'timeline': 'Ongoing'
    })

# Print action plan
for action in action_plan:
    print(f"\n{action['priority']}. {action['action']}")
    print(f"   Details: {action['details']}")
    print(f"   Recommended: {action['recommended']}")
    print(f"   Timeline: {action['timeline']}")

print("\n" + "=" * 60)

---

## Summary

This analysis has provided:
- ✅ Complete portfolio overview with performance metrics
- ✅ Risk assessment including concentration and diversification analysis
- ✅ Trading signals based on momentum and risk management
- ✅ Visual representations of portfolio composition
- ✅ Prioritized action plan for portfolio optimization

### Key Takeaways

1. **Portfolio Health**: Review the risk metrics to ensure your portfolio aligns with your risk tolerance
2. **Diversification**: Check if your concentration risk is within acceptable levels
3. **Performance**: Monitor your best and worst performers regularly
4. **Rebalancing**: Consider rebalancing quarterly to maintain your target allocation
5. **Cash Management**: Maintain adequate cash reserves for opportunities and emergencies

### Next Steps

1. Review the action plan above
2. Execute priority 1 actions first
3. Monitor portfolio daily for price movements
4. Run this analysis weekly to track progress
5. Adjust strategy based on market conditions

### Important Disclaimer

This analysis is for informational purposes only and does not constitute financial advice. Always conduct your own due diligence and consult with a financial advisor before making investment decisions.

---

**Analysis completed at**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}